# Assignment 8: Build the Intent Dataset for Your Own Chatbot
### AITCD001 Chatbots Development · Learning Outcome 3 · United African Institute of Technology


Cells marked **YOUR DOMAIN** are replaced with your own chatbot's data. Cells marked **PROVIDED**
do the checking, the split, the baseline and the entity count: read them, do not change them.

The worked example is a **fictional** bus ticket desk (the Week 7 example) with only 6 messages
per label, so the quality report shows what is still missing. It may not be submitted.

**Steps:** Save a copy of this file in Drive. Replace Steps 1, 2, 3, 6 and 7 with your own, and run
everything from the top (Runtime, then Run all). Repeat Steps 3 and 4 until the report is clean.

---

---

## Setup

**PROVIDED.** Run once. The same spaCy model as the session notebook.

In [ ]:

import csv
import random
import re
from collections import Counter

import pandas as pd
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:                  # not installed yet: install it once
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("ready: spaCy", spacy.__version__, "| model", nlp.meta["version"])

ready: spaCy 3.8.18 | model 3.8.0


---

## Step 1: The chatbot and its labels

**YOUR DOMAIN.** The chatbot of your Portfolio Project 1. All of its intents, plus `out_of_scope`. Do not add `greeting` or `name` unless your users really ask for something with them.

In [ ]:
# STEP 1. YOUR DOMAIN: the chatbot of my Portfolio Project 1, and its labels.
BOT_NAME = "Clinic Appointment Assistant"
INTENTS = ["hours", "prices", "services", "location", "book", "cancel", "emergency", "out_of_scope"]

TARGET = 30          # messages per label, at least
print(BOT_NAME, "|", len(INTENTS), "labels:", INTENTS)

Clinic Appointment Assistant | 8 labels: ['hours', 'prices', 'services', 'location', 'book', 'cancel', 'emergency', 'out_of_scope']


---

## Step 2: The labelling guideline

**YOUR DOMAIN.** Write the guideline BEFORE labelling. Every time two labels are possible for a message, add or sharpen a rule.

In [ ]:
# STEP 2. YOUR DOMAIN: my labelling guideline.
# Each rule settles a case where two people could disagree. Copy it into the README.
GUIDELINE = [
    "One message, one label. Two requests in one message: split it first.",
    "Label the goal, not the words: 'is dental expensive' is prices, not services.",
    "Asking what the clinic offers is services. Asking what it costs is prices, even if a service is named.",
    "Wanting a visit, a doctor or a slot is book, even with a day or a service in it.",
    "Stopping a visit or a booking is cancel. Changing it to another day is out_of_scope: the bot can only book or cancel.",
    "Anything urgent about a person's body or safety (bleeding, breathing, accident) is emergency, even if it also asks for a visit.",
    "Medical advice, test results, insurance or MoMo payment, complaints, jobs, lost property and chat: out_of_scope.",
    "Keep the patient's words: typos, SMS, Kinyarwanda, as typed.",
]

for i, rule in enumerate(GUIDELINE, start=1):
    print(i, rule)

1 One message, one label. Two requests in one message: split it first.
2 Label the goal, not the words: 'is dental expensive' is prices, not services.
3 Asking what the clinic offers is services. Asking what it costs is prices, even if a service is named.
4 Wanting a visit, a doctor or a slot is book, even with a day or a service in it.
5 Stopping a visit or a booking is cancel. Changing it to another day is out_of_scope: the bot can only book or cancel.
6 Anything urgent about a person's body or safety (bleeding, breathing, accident) is emergency, even if it also asks for a visit.
7 Medical advice, test results, insurance or MoMo payment, complaints, jobs, lost property and chat: out_of_scope.
8 Keep the patient's words: typos, SMS, Kinyarwanda, as typed.


---

## Step 3: The dataset

**YOUR DOMAIN.** At least 30 messages per label. Sources, in order of value: your chatbot's UNMATCHED log and Step 10 local phrasing from Portfolio Project 1; at least 20 messages from real people you ask (classmates, family, friends, for example on WhatsApp: *how would you ask a bus desk about luggage?*); then your own. Keep every message exactly as it was typed.

In [ ]:
# STEP 3. YOUR DOMAIN: the dataset. Each row is (message, label, source).
# source is one of: "rules", "unmatched", "local", "people", "written"
# "people" rows are messages from REAL people I asked (see PEOPLE below).
DATA = [
    ('what time do you open', 'hours', 'rules'),
    ('are you open on saturday', 'hours', 'written'),
    ('wat tym u open', 'hours', 'local'),
    ('mufungura saa ngapi', 'hours', 'local'),
    ('when do you close', 'hours', 'written'),
    ('are you open on sunday', 'hours', 'written'),
    ('opening hours please', 'hours', 'written'),
    ('amasaha mukorera ni ayahe', 'hours', 'written'),
    ('mukora saa ngapi', 'hours', 'written'),
    ('do you work on public holidays', 'hours', 'written'),
    ('what time do you close on friday', 'hours', 'written'),
    ('opn tmrw?', 'hours', 'written'),
    ('hours pls', 'hours', 'written'),
    ('are you open now', 'hours', 'written'),
    ('mufungura ku cyumweru?', 'hours', 'written'),
    ('what time does the clinic start', 'hours', 'written'),
    ('is the clinic open today', 'hours', 'written'),
    ('closing time', 'hours', 'written'),
    ('saa ngahe mufunga', 'hours', 'written'),
    ('open 24 hours?', 'hours', 'written'),
    ('are you open on weekends', 'hours', 'written'),
    ('what are your working hours', 'hours', 'written'),
    ('mukora ku wa gatandatu?', 'hours', 'written'),
    ('wat time u close 2day', 'hours', 'written'),
    ('clinic hours on saturday', 'hours', 'written'),
    ('mufunguye none?', 'hours', 'written'),
    ('do you close for lunch', 'hours', 'written'),
    ('whn do u open', 'hours', 'written'),
    ('how late are you open', 'hours', 'written'),
    ('amasaha yo gufungura', 'hours', 'written'),
    ('how much is dental', 'prices', 'rules'),
    ('hw much 4 dental', 'prices', 'local'),
    ('amenyo ni angahe', 'prices', 'rules'),
    ('how much is a consultation', 'prices', 'written'),
    ('what does a lab test cost', 'prices', 'written'),
    ('ibiciro ni ibihe', 'prices', 'written'),
    ('price for general consultation', 'prices', 'written'),
    ('how much for lab', 'prices', 'written'),
    ('consultation fee?', 'prices', 'written'),
    ('kwisuzumisha bigura angahe', 'prices', 'written'),
    ('cost of seeing the dentist', 'prices', 'written'),
    ('hw much 4 lab test', 'prices', 'written'),
    ('dental price pls', 'prices', 'written'),
    ('how much do i pay for consultation', 'prices', 'written'),
    ('what are your fees', 'prices', 'written'),
    ('prices', 'prices', 'written'),
    ('lab ni angahe', 'prices', 'written'),
    ('how much is it to see a doctor', 'prices', 'written'),
    ('tell me the price of teeth cleaning', 'prices', 'written'),
    ('how much will the lab cost me', 'prices', 'written'),
    ('consultation ni angahe', 'prices', 'written'),
    ('ibiciro bya laboratwari', 'prices', 'written'),
    ('wat is the cost of general', 'prices', 'written'),
    ('is dental expensive', 'prices', 'written'),
    ('how much 2 see dentist', 'prices', 'written'),
    ('fees for lab test', 'prices', 'written'),
    ("ikiguzi cy'amenyo", 'prices', 'written'),
    ('mwishyuza angahe kuri consultation', 'prices', 'written'),
    ('price list', 'prices', 'written'),
    ('cost of a check up', 'prices', 'written'),
    ('what services do you offer', 'services', 'rules'),
    ('do you have a dentist', 'services', 'written'),
    ('mufite serivisi ki', 'services', 'written'),
    ('do you do lab tests', 'services', 'written'),
    ('which services are available', 'services', 'written'),
    ('mwivura iki', 'services', 'written'),
    ('do you treat children', 'services', 'written'),
    ('do you offer x-ray', 'services', 'written'),
    ('serivisi mutanga', 'services', 'written'),
    ('do you have a laboratory', 'services', 'written'),
    ('wat services u hav', 'services', 'written'),
    ('do you do blood tests', 'services', 'written'),
    ('mufite amenyo', 'services', 'written'),
    ('can i do a malaria test there', 'services', 'written'),
    ('do you do vaccinations', 'services', 'written'),
    ('what do you treat', 'services', 'written'),
    ('ese mufite laboratwari', 'services', 'written'),
    ('services list pls', 'services', 'written'),
    ('do you have a pharmacy', 'services', 'written'),
    ('are there specialists', 'services', 'written'),
    ('do you do teeth cleaning', 'services', 'written'),
    ('mukora iki', 'services', 'written'),
    ('do you do pregnancy tests', 'services', 'written'),
    ('what can i get at your clinic', 'services', 'written'),
    ("mufite umuganga w'abana", 'services', 'written'),
    ('do you have an eye doctor', 'services', 'written'),
    ('what kind of tests do you do', 'services', 'written'),
    ('do you offer general consultation', 'services', 'written'),
    ('which dental services do you have', 'services', 'written'),
    ('serivisi zanyu', 'services', 'written'),
    ('where are you', 'location', 'rules'),
    ('where is the clinic', 'location', 'written'),
    ('what is your address', 'location', 'written'),
    ('muri he', 'location', 'written'),
    ('muherereye he', 'location', 'written'),
    ('clinic location pls', 'location', 'written'),
    ('how do i get to kacyiru', 'location', 'written'),
    ('wea r u', 'location', 'written'),
    ('which street are you on', 'location', 'written'),
    ('address?', 'location', 'written'),
    ('aho muherereye', 'location', 'local'),
    ('how do i find you', 'location', 'written'),
    ('where is the clinic in kigali', 'location', 'written'),
    ('mwanyereka aho muri', 'location', 'written'),
    ('are you in kacyiru', 'location', 'written'),
    ('location', 'location', 'written'),
    ('wer is ur clinic', 'location', 'written'),
    ('can u send the location', 'location', 'written'),
    ('is the clinic far from town', 'location', 'written'),
    ('muri kacyiru?', 'location', 'written'),
    ('what is the nearest bus stop', 'location', 'written'),
    ('where are u located', 'location', 'written'),
    ('ikigo cyanyu giherereye he', 'location', 'written'),
    ('directions to the clinic', 'location', 'written'),
    ('do you have a map', 'location', 'written'),
    ('ur address pls', 'location', 'written'),
    ('which district are you in', 'location', 'written'),
    ('is it on kg 567 st', 'location', 'written'),
    ('how far is the clinic from nyabugogo', 'location', 'written'),
    ('mwampa aho muherereye', 'location', 'written'),
    ('i want to book a visit', 'book', 'rules'),
    ('book an appointment', 'book', 'rules'),
    ('book visit 2moro', 'book', 'local'),
    ('ndashaka kwivuza', 'book', 'local'),
    ('nshaka umuganga', 'book', 'unmatched'),
    ('i need an appointment for dental', 'book', 'written'),
    ('can i reserve a slot for friday', 'book', 'written'),
    ('ndashaka gahunda', 'book', 'written'),
    ('i want to see a doctor tomorrow', 'book', 'written'),
    ('book me for lab test on monday', 'book', 'written'),
    ('kwiyandikisha', 'book', 'written'),
    ('i would like to make an appointment', 'book', 'written'),
    ('book a dental visit', 'book', 'written'),
    ('reserve a consultation', 'book', 'written'),
    ('nshaka kwivuza ku wa mbere', 'book', 'written'),
    ('can i come on saturday morning', 'book', 'written'),
    ('bk appointment pls', 'book', 'written'),
    ('i want to see the dentist', 'book', 'written'),
    ('need to see a doctor today', 'book', 'written'),
    ('book consultation for tuesday afternoon', 'book', 'written'),
    ('ndashaka kubonana na muganga', 'book', 'written'),
    ('schedule a visit', 'book', 'written'),
    ('can i get an appointment this week', 'book', 'written'),
    ('i want to do a lab test tomorrow', 'book', 'written'),
    ('gahunda ya dentist', 'book', 'written'),
    ('set an appointment for my son', 'book', 'written'),
    ('make me an appointment', 'book', 'written'),
    ('kwivuza ejo', 'book', 'written'),
    ('get me a slot on thursday', 'book', 'written'),
    ('i need to visit the clinic', 'book', 'written'),
    ('cancel', 'cancel', 'rules'),
    ('cancle', 'cancel', 'local'),
    ('cancel my appointment', 'cancel', 'written'),
    ('ndashaka guhagarika gahunda', 'cancel', 'written'),
    ('i cant come anymore', 'cancel', 'written'),
    ('call it off', 'cancel', 'written'),
    ('cancel the visit for friday', 'cancel', 'written'),
    ('sinshobora kuza', 'cancel', 'written'),
    ('hagarika', 'cancel', 'written'),
    ('i want to cancel my booking', 'cancel', 'written'),
    ('please cancel my dental', 'cancel', 'written'),
    ('stop the appointment', 'cancel', 'written'),
    ('cancl my appt', 'cancel', 'written'),
    ('nta kuza', 'cancel', 'written'),
    ('i wont make it tomorrow', 'cancel', 'written'),
    ('guhagarika gahunda yanjye', 'cancel', 'written'),
    ('remove my booking', 'cancel', 'written'),
    ('i changed my mind, dont book', 'cancel', 'written'),
    ('cancel lab test', 'cancel', 'written'),
    ('no longer need the appointment', 'cancel', 'written'),
    ('ndahagaritse', 'cancel', 'written'),
    ('cancel monday visit', 'cancel', 'written'),
    ('i need to cancel', 'cancel', 'written'),
    ('i am sick and cant come', 'cancel', 'written'),
    ('cancel pls', 'cancel', 'written'),
    ('undo my booking', 'cancel', 'written'),
    ('sinkibonetse kuza', 'cancel', 'written'),
    ('cancel everything', 'cancel', 'written'),
    ('i will not come on saturday', 'cancel', 'written'),
    ('withdraw my appointment request', 'cancel', 'written'),
    ('emergency', 'emergency', 'written'),
    ('i have an emergency', 'emergency', 'written'),
    ('bleeding', 'emergency', 'written'),
    ('she is bleeding a lot', 'emergency', 'written'),
    ('i cant breathe', 'emergency', 'written'),
    ('amaraso', 'emergency', 'written'),
    ('ihutirwa', 'emergency', 'rules'),
    ('help my child is not breathing', 'emergency', 'written'),
    ('ndi kuva amaraso', 'emergency', 'written'),
    ('ambulance please', 'emergency', 'written'),
    ('i think i am having a heart attack', 'emergency', 'written'),
    ('there is an accident', 'emergency', 'written'),
    ('urgent help', 'emergency', 'written'),
    ('bleeding badly', 'emergency', 'written'),
    ('my father fell and is not responding', 'emergency', 'written'),
    ('ndi guhumeka nabi', 'emergency', 'written'),
    ('umwana arimo guhumeka nabi', 'emergency', 'written'),
    ('emergency pls', 'emergency', 'written'),
    ('i cant breath', 'emergency', 'written'),
    ('someone fainted', 'emergency', 'written'),
    ('uwarwaye cyane', 'emergency', 'written'),
    ("mfite ikibazo cy'ihutirwa", 'emergency', 'written'),
    ('my wife is in labour and bleeding', 'emergency', 'written'),
    ('chest pain, very strong', 'emergency', 'written'),
    ('he is unconscious', 'emergency', 'written'),
    ('ndwaye cyane ubu', 'emergency', 'written'),
    ('it is urgent', 'emergency', 'written'),
    ('i was bitten by a snake', 'emergency', 'written'),
    ('ihutirwa mufashe', 'emergency', 'written'),
    ('heavy bleeding after accident', 'emergency', 'written'),
    ('who won the match', 'out_of_scope', 'written'),
    ('asdfgh', 'out_of_scope', 'unmatched'),
    ('qwerty', 'out_of_scope', 'unmatched'),
    ('do you accept mutuelle insurance', 'out_of_scope', 'written'),
    ('can i pay with momo', 'out_of_scope', 'written'),
    ('are you hiring nurses', 'out_of_scope', 'written'),
    ('the doctor was rude', 'out_of_scope', 'written'),
    ('i lost my phone at the clinic', 'out_of_scope', 'written'),
    ('what is the weather in kigali', 'out_of_scope', 'written'),
    ('give me medicine for headache', 'out_of_scope', 'written'),
    ('what is my test result', 'out_of_scope', 'written'),
    ('i want to change my appointment to friday', 'out_of_scope', 'written'),
    ('tell me a joke', 'out_of_scope', 'written'),
    ('how do i lose weight', 'out_of_scope', 'written'),
    ('i have a headache what should i do', 'out_of_scope', 'written'),
    ('is malaria dangerous', 'out_of_scope', 'written'),
    ('send me my medical records', 'out_of_scope', 'written'),
    ('job vacancies?', 'out_of_scope', 'written'),
    ('i want to complain', 'out_of_scope', 'written'),
    ('hmm', 'out_of_scope', 'written'),
    ('pls help me', 'out_of_scope', 'unmatched'),
    ('what is the capital of rwanda', 'out_of_scope', 'written'),
    ('i want to pay my bill', 'out_of_scope', 'written'),
    ('do you sell phones', 'out_of_scope', 'written'),
    ('cheap flights to nairobi', 'out_of_scope', 'written'),
    ('nibyo', 'out_of_scope', 'written'),
    ('where can i buy a bus ticket', 'out_of_scope', 'written'),
    ('my results are wrong', 'out_of_scope', 'written'),
    ('how old is the clinic', 'out_of_scope', 'written'),
    ('the nurse was late', 'out_of_scope', 'written'),
]

# PEOPLE: paste at least 20 messages from real people you asked (classmates, family, friends).
# Format: ("exact message as typed", "label"). Remove any name, phone number or ID number.
# Aim for 3 or 4 per label, with Kinyarwanda or SMS phrasing in every intent.
PEOPLE = [
    ("What time does the clinic open?", "hours"),
    ("Guhura na muganga bisaba angahe?", "prices"),
    ("Where is your clinic located?", "location"),
    ("Ese nshobora gufata gahunda yo kuza ejo?", "book"),
    ("What services does your clinic offer?", "services"),
    ("Ivuriro rifunga saa ngahe?", "hours"),
    ("How can I cancel my appointment?", "cancel"),
    ("Ese mutanga serivisi zo kuvura amenyo?", "services"),
    ("How much does a blood test cost?", "prices"),
    ("Nakora iki mu gihe habaye ikibazo cyihutirwa cy'ubuzima?", "emergency"),
    ("Is the clinic open on Sundays?", "hours"),
    ("Ese nshobora guhindura itariki ya gahunda yanjye?", "out_of_scope"),
    ("Do you provide maternity services?", "services"),
    ("Kwisuzumisha muri rusange bisaba angahe?", "prices"),
    ("Can I book an appointment online?", "book"),
    ("Ese mwakira abarwayi batabanje gufata gahunda?", "book"),
    ("What should I do if someone has difficulty breathing?", "emergency"),
    ("Ese mufite serivisi z'imbangukiragutabara?", "services"),
    ("Can I get a refund if I cancel my appointment?", "out_of_scope"),
    ("Is there a clinic near the city centre?", "location"),
]
for text, label in PEOPLE:
    DATA.append((text, label, "people"))

print(len(DATA), "rows")

260 rows


---

## Step 4: The quality report

**PROVIDED.** Read every line. Fix the data in Step 3 until every label is ok, there are no duplicates and no conflicts, and the balance is 1.5 or less. Then run Step 3 and Step 4 again.

In [ ]:
# STEP 4. PROVIDED: the quality report. Read it, fix the data in Step 3, run again.
def clean(text):
    """Remove punctuation (keeping the colon of 21:00), lowercase, trim."""
    return re.sub(r"[^\w\s:]", "", text).lower().strip()


per_label = Counter()
per_source = Counter()
unknown = []
for text, label, source in DATA:
    per_label[label] += 1
    per_source[source] += 1
    if label not in INTENTS:
        unknown.append((text, label))

print("Per label:")
for label in INTENTS:
    n = per_label[label]
    note = "ok" if n >= TARGET else f"needs more, target {TARGET}"
    print(f"   {label:14}{n:4}   {note}")
print("Per source:", dict(per_source))
print("people:", per_source["people"], "of at least 20")

seen = {}
duplicates = []
conflicts = []
for text, label, source in DATA:
    key = clean(text)
    if key in seen:
        duplicates.append(text)
        if seen[key] != label:
            conflicts.append(text)
    seen[key] = label

print("Labels not in INTENTS:", unknown)
print("Duplicates:", duplicates)
print("Conflicting labels:", conflicts)
counts = [per_label[label] for label in INTENTS]
print("Balance, largest / smallest:", round(max(counts) / max(1, min(counts)), 2))

Per label:
   hours           33   ok
   prices          33   ok
   services        34   ok
   location        32   ok
   book            33   ok
   cancel          31   ok
   emergency       32   ok
   out_of_scope    32   ok
Per source: {'rules': 9, 'written': 220, 'local': 7, 'unmatched': 4, 'people': 20}
people: 20 of at least 20
Labels not in INTENTS: []
Duplicates: []
Conflicting labels: []
Balance, largest / smallest: 1.1


---

## Step 5: The split, and the CSV file

**PROVIDED.** 20% of each label becomes the test set. The file `intents.csv` appears in the Files panel on the left: download it, and commit it to GitHub.

In [ ]:
# STEP 5. PROVIDED: the 80/20 split, intent by intent, with a fixed seed; then the CSV.
rng = random.Random(8)

by_label = {}
for row in DATA:
    if row[1] not in by_label:
        by_label[row[1]] = []
    by_label[row[1]].append(row)

TRAIN = []
TEST = []
for label in sorted(by_label):
    rows = by_label[label][:]
    rng.shuffle(rows)
    cut = len(rows) // 5
    TEST.extend(rows[:cut])
    TRAIN.extend(rows[cut:])

with open("intents.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["text", "intent", "source", "split"])
    for text, label, source in TRAIN:
        writer.writerow([text, label, source, "train"])
    for text, label, source in TEST:
        writer.writerow([text, label, source, "test"])

df = pd.read_csv("intents.csv")
print(len(TRAIN), "training,", len(TEST), "test, saved to intents.csv")
print(pd.crosstab(df["intent"], df["split"]))

212 training, 48 test, saved to intents.csv
split         test  train
intent                   
book             6     27
cancel           6     25
emergency        6     26
hours            6     27
location         6     26
out_of_scope     6     26
prices           6     27
services         6     28


---

## Step 6: The baseline: your rules on your test set

**YOUR DOMAIN.** Paste the rule table of your own chatbot. The baseline is the number your classifier must beat in Week 9. Every line it prints is a message the rules get wrong.

In [ ]:
# STEP 6. YOUR DOMAIN: the RULES list from Step 2 of my Portfolio Project 1.
# (name and greeting are not labels in this dataset, so they are left out.)
RULES = [
    ("emergency", r"\bemergency\b|\bbleeding\b|\bcan ?t breathe\b|\bamaraso\b|\bihutirwa\b"),
    ("cancel",    r"\bcanc\w*\b|\bhagarika\w*\b|\bcall it off\b"),
    ("hours",     r"\bhours?\b|\bopen\w*\b|\bclos\w*\b|\bwhat ?time\b|\bwat ?tym\b|\bsaa ngapi\b|\bamasaha\b|\bgufungura\b"),
    ("prices",    r"\bprices?\b|\bcosts?\b|\bfees?\b|\bhow ?much\b|\bhw ?much\b|\bangahe\b|\bibiciro\b"),
    ("services",  r"\bservices?\b|\bserivisi\b|\boffer\w*\b|\bmwivura\b|\bmufite\b"),
    ("location",  r"\bwhere\b|\blocat\w*\b|\baddress\b|\bmuri he\b|\bmuherereye\b"),
    ("book",      r"\bbook\w*\b|\bappointments?\b|\bvisit\b|\breserv\w*\b|\bgahunda\b|\bkwiyandikisha\b|\bkwivuza\b"),
]
print(len(RULES), "rules")

7 rules


In [ ]:
# STEP 6, continued. PROVIDED: the baseline, your rules on your test set.
def rule_label(text):
    """The first rule that matches, or out_of_scope."""
    for label, pattern in RULES:
        if re.search(pattern, clean(text)):
            return label
    return "out_of_scope"


right = 0
for text, label, source in TEST:
    guess = rule_label(text)
    if guess == label:
        right += 1
    else:
        print(f"   {text!r}: labelled {label}, rules say {guess}")
print(f"Baseline: {right} of {len(TEST)} test messages right")

   'i want to see a doctor tomorrow': labelled book, rules say out_of_scope
   'need to see a doctor today': labelled book, rules say out_of_scope
   'stop the appointment': labelled cancel, rules say book
   'i cant come anymore': labelled cancel, rules say out_of_scope
   'nta kuza': labelled cancel, rules say out_of_scope
   'i will not come on saturday': labelled cancel, rules say out_of_scope
   'remove my booking': labelled cancel, rules say book
   'help my child is not breathing': labelled emergency, rules say out_of_scope
   'chest pain, very strong': labelled emergency, rules say out_of_scope
   "mfite ikibazo cy'ihutirwa": labelled emergency, rules say out_of_scope
   'What should I do if someone has difficulty breathing?': labelled emergency, rules say out_of_scope
   'muri kacyiru?': labelled location, rules say out_of_scope
   'wea r u': labelled location, rules say out_of_scope
   'which street are you on': labelled location, rules say out_of_scope
   'i want to change m

---

## Step 7: The entities of your domain

**YOUR DOMAIN.** All entity labels of your own, each with the closed list of names your users mention: places, products, services, days. The provided part counts how often they appear in your dataset.

In [ ]:
# STEP 7. YOUR DOMAIN: the entities of my domain. At least 2 labels of my own.
ENTITY_LISTS = {
    "SERVICE": ["consultation", "dental", "dentist", "lab", "laboratory", "laboratwari", "amenyo"],
    "DAY": ["today", "tomorrow", "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"],
    "PLACE": ["kacyiru", "kigali", "nyabugogo"],
}

patterns = []
for label in ENTITY_LISTS:
    for name in ENTITY_LISTS[label]:
        patterns.append({"label": label, "pattern": [{"LOWER": name}]})

if "entity_ruler" in nlp.pipe_names:
    nlp.remove_pipe("entity_ruler")
ruler = nlp.add_pipe("entity_ruler", before="ner")
ruler.add_patterns(patterns)

# PROVIDED: how many of your messages contain each entity, and five examples
found = Counter()
examples = []
for text, label, source in DATA:
    ents = [(e.text, e.label_) for e in nlp(text.lower()).ents]
    for e in ents:
        found[e[1]] += 1
    if ents and len(examples) < 5:
        examples.append((text, ents))
print("Entities found in the dataset:", dict(found))
for text, ents in examples:
    print("  ", text, "->", ents)

Entities found in the dataset: {'DAY': 18, 'PERSON': 30, 'TIME': 6, 'ORG': 8, 'DATE': 2, 'SERVICE': 38, 'CARDINAL': 4, 'GPE': 6, 'PLACE': 6, 'NORP': 1}
   are you open on saturday -> [('saturday', 'DAY')]
   mufungura saa ngapi -> [('mufungura', 'PERSON')]
   are you open on sunday -> [('sunday', 'DAY')]
   opening hours please -> [('opening hours', 'TIME')]
   amasaha mukorera ni ayahe -> [('amasaha mukorera ni', 'ORG')]
